# Notebook 15 — Set Operations

**Datasets:** `samples.bakehouse.sales_customers`, `samples.tpch.customer`, `samples.wanderbricks.users`  
**Difficulty:** Easy  
**Topics:** `union`, `unionByName`, `intersect`, `subtract`, `distinct`, `dropDuplicates`, `except`

Set operations let you combine, compare, and deduplicate DataFrames — essential for data reconciliation and multi-source pipelines.

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

customers = spark.read.table("samples.bakehouse.sales_customers")
tpch_customer = spark.read.table("samples.tpch.customer")
users = spark.read.table("samples.wanderbricks.users")

print("sales_customers schema:")
customers.printSchema()
print("tpch.customer schema:")
tpch_customer.printSchema()
print("wanderbricks.users schema:")
users.printSchema()

## Learn — Set Operations

| Function | What it does |
|----------|-------------|
| `df1.union(df2)` | Stack rows from df2 below df1 (columns must match by position) |
| `df1.unionByName(df2)` | Stack rows matching by column name (safer) |
| `df1.intersect(df2)` | Rows that appear in BOTH DataFrames |
| `df1.subtract(df2)` | Rows in df1 that do NOT appear in df2 |
| `df.distinct()` | Remove exact duplicate rows |
| `df.dropDuplicates(["col1", "col2"])` | Remove duplicates based on specific columns only |

**Docs:** [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

> `union` matches columns by **position**, not name — column order must be identical.
> `unionByName` matches by **name** and is generally safer and more explicit.

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

# Simulate two batches of data to demonstrate union
batch1 = spark.createDataFrame([("Alice", "UK"), ("Bob", "US")], ["name", "country"])
batch2 = spark.createDataFrame([("Carol", "AU"), ("Alice", "UK")], ["name", "country"])

# union (with duplicates)
combined = batch1.union(batch2)
print("union:", combined.count(), "rows")

# distinct removes the duplicate Alice/UK row
print("after distinct:", combined.distinct().count(), "rows")

# subtract: who is in batch2 but NOT in batch1?
only_in_batch2 = batch2.subtract(batch1)
only_in_batch2.show()

## Problem 1 — Union Two DataFrames

Build a unified list of countries from two sources:
1. From `sales_customers`: get distinct `country` values, add a `source` column with literal `'bakehouse'`
2. From `wanderbricks.users`: get distinct `country` values, add a `source` column with literal `'wanderbricks'`

Combine them using `.unionByName()`. Each row represents one country-source combination.


**Expected output columns:** `country`, `source`

In [0]:
distinct_cntry_bh = customers.select("country").distinct().withColumn("source", F.lit("bakehouse"))
distinct_cntry_wu = users.select("country").distinct().withColumn("source", F.lit("wanderbricks"))
result_1 = distinct_cntry_bh.unionByName(distinct_cntry_wu)
result_1.display()

In [0]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'source' in cols, "Missing column: source"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Find Common Countries

Use `.intersect()` to find countries that appear in **both** `sales_customers` and `wanderbricks.users`.


**Expected output columns:** `country`

In [0]:
result_2 = distinct_cntry_bh.select("country").intersect(distinct_cntry_wu.select("country"))

In [0]:
result_2.display()

In [0]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'country' in cols, "Missing column: country"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Find Countries Unique to Bakehouse

Use `.subtract()` to find countries that are in `sales_customers` but **NOT** in `wanderbricks.users`.


**Expected output columns:** `country`

In [0]:
result_3 = distinct_cntry_bh.select("country").subtract(distinct_cntry_wu.select("country"))

In [0]:
result_3.display()

In [0]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'country' in cols, "Missing column: country"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt >= 0, f"Unexpected negative row count"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Remove Duplicates

On `sales_customers`, treat `(first_name, last_name, country)` as a natural key. Use `.dropDuplicates(["first_name", "last_name", "country"])` to keep only one row per combination.

Return a summary DataFrame showing the row count **before** and **after** deduplication.


**Expected output columns:** `step`, `row_count`

In [0]:
before = customers.count()
after = customers.dropDuplicates(["first_name", "last_name", "country"]).count()
rows = [("before", before), ("after", after)]
result_4 = spark.createDataFrame(rows, ["step", "row_count"])
result_4.display()

In [0]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'step' in cols, "Missing column: step"
assert 'row_count' in cols, "Missing column: row_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — Union All Three User Sources

Build a unified user list from two datasets using a consistent schema:

1. From `sales_customers`: `customerID` as `user_id`, `first_name` as `name`, `country`, `F.lit('bakehouse')` as `source`
2. From `wanderbricks.users`: `user_id`, `name`, `country`, `F.lit('wanderbricks')` as `source`

Union them with `.unionByName()`, then group by `source` to count users per source.

**Expected output columns:** `source`, `user_count`

In [0]:
result_5 = customers.select(
    F.col("customerID").alias("user_id"),
    F.col("first_name").alias("name"),
    "country",
    F.lit("bakehouse").alias("source")
).unionByName(
    users.select(
        "user_id",
        "name",
        "country",
        F.lit("wanderbricks").alias("source")
    )
).groupBy("source").agg(F.count("*").alias("user_count"))

result_5.display()

In [0]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'source' in cols, "Missing column: source"
assert 'user_count' in cols, "Missing column: user_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")